In [1]:
print("ASG Airlines Data Engineering Project")
print("Notebook started successfully.")

ASG Airlines Data Engineering Project
Notebook started successfully.


# ASG Airlines Data Engineering Project

## Project Overview

This project focuses on building a data engineering and business intelligence workflow using airline flight, booking, passenger, and payment data.

The project includes data extraction, data cleaning, data quality validation, database creation, SQL analysis, and Power BI visualization. The cleaned data is organized into a structured data model to support analysis of flights, bookings, routes, payments, and data quality.

### Objectives

- Extract data from the provided Excel workbook.
- Clean and validate the airline datasets.
- Identify missing, invalid, duplicate, and inconsistent records.
- Store the cleaned data in a SQLite database.
- Create a star-schema-based analytical data model.
- Perform SQL-based analysis and calculate key performance indicators (KPIs).
- Generate analytical datasets for Power BI.
- Build an interactive Power BI dashboard for business insights.

In [2]:
import pandas as pd
import numpy as np
import sqlite3
from pathlib import Path

## 1. Data Loading

The airline dataset is provided as an Excel workbook containing four related datasets: flights, payments, bookings, and passengers. The workbook is loaded using Pandas for further exploration and preprocessing.

In [3]:
file = "../data/raw/UseCase - Airlines.xlsx"

excel_data = pd.ExcelFile(file)

print("Available sheets:")
print(excel_data.sheet_names)

Available sheets:
['flights', 'payments', 'bookings', 'passengers']


In [4]:
flights = pd.read_excel(file, sheet_name="flights")
payments = pd.read_excel(file, sheet_name="payments")
bookings = pd.read_excel(file, sheet_name="bookings")
passengers = pd.read_excel(file, sheet_name="passengers")

print("Flights:", flights.shape)
print("Payments:", payments.shape)
print("Bookings:", bookings.shape)
print("Passengers:", passengers.shape)

Flights: (1020, 7)
Payments: (1000, 4)
Bookings: (1000, 9)
Passengers: (1039, 9)


In [5]:
print("FLIGHTS")
display(flights.head())

print("PAYMENTS")
display(payments.head())

print("BOOKINGS")
display(bookings.head())

print("PASSENGERS")
display(passengers.head())

FLIGHTS


,flight_id,airline,source,destination,departure_time,arrival_time,duration
0,SJ010,SpiceJet,CCU,MAA,2026-04-20 23:38:41.701,2026-04-21 02:32:41.701,02:54:00
1,AI155,Air India,BOM,CCU,2026-04-20 23:35:41.703,2026-04-21 01:23:41.703,01:48:00
2,UK094,Vistara,BOM,CCU,2026-04-20 23:26:41.702,2026-04-21 01:11:41.702,01:45:00
3,AI245,Air India,BOM,CCU,2026-04-20 23:07:41.704,2026-04-21 01:43:41.704,02:36:00
4,AI192,Air India,MAA,BOM,2026-04-20 23:05:41.703,2026-04-21 04:04:41.703,04:59:00


PAYMENTS


,payment_id,booking_id,amount,payment_method
0,PAY1000,B1116,9883.49,NETBANKING
1,PAY1001,B1738,8457.96,NETBANKING
2,PAY1002,B1873,6495.37,UPI
3,PAY1003,B1914,5079.38,NETBANKING
4,PAY1004,B1967,12518.31,CARD


BOOKINGS


,booking_id,passenger_id,flight_id,booking_date,status,passport_number,seat_number,emergency_contact_name,emergency_contact_phone
0,B1000,P1591,AI192,2025-06-14 11:37:36.951,CANCELLED,P1945887,3D,Isaac Bakshi,+91-6478475128
1,B1001,P1803,6F026,2025-11-02 11:37:36.951,CANCELLED,L3482012,18A,Anvi Konda,+91-6647078662
2,B1002,P1083,SJ010,2025-08-25 11:37:36.951,CANCELLED,G8507659,30C,Udant Dewan,+91-8405938220
3,B1003,P1364,AI069,2025-12-30 11:37:36.951,CONFIRMED,M0891776,33A,Harsh Chahal,+91-6264636839
4,B1004,P1885,UK003,2025-10-02 11:37:36.951,PENDING,N5742231,25C,Pahal Balay,+91-9336478266


PASSENGERS


,passenger_id,first_name,last_name,age,gender,email,phone,aadhaar_id,date_of_birth
0,P1000,Vivaan,Chatterjee,52,F,vivaan.chatterjee@gmail.com,+91-6896233790,433218196001,1974-04-08
1,P1001,Krishna,Reddy,15,M,krishna.reddy@hotmail.com,+91-6702632297,386379402654,2011-03-07
2,P1002,Myra,Naidu,72,M,myra.naidu@outlook.com,+91-6199585092,615594078161,1954-09-10
3,P1003,Myra,Mishra,61,F,myra.mishra@hotmail.com,+91-8719927151,310341316475,1965-03-12
4,P1004,Saanvi,Banerjee,21,M,saanvi.banerjee@outlook.com,+91-7819595113,419283276483,2005-11-11


## 2. Data Exploration and Quality Assessment

The loaded datasets are examined to understand their structure, identify missing values, and detect duplicate records before performing the cleaning process.

In [6]:
print("Missing values in Flights:")
display(flights.isnull().sum())

print("Missing values in Payments:")
display(payments.isnull().sum())

print("Missing values in Bookings:")
display(bookings.isnull().sum())

print("Missing values in Passengers:")
display(passengers.isnull().sum())

Missing values in Flights:


flight_id          0
airline           41
source             0
destination        0
departure_time     0
arrival_time       0
duration           0
dtype: int64

Missing values in Payments:


payment_id         0
booking_id         0
amount            48
payment_method     0
dtype: int64

Missing values in Bookings:


booking_id                  0
passenger_id                0
flight_id                   0
booking_date                0
status                     45
passport_number             0
seat_number                 0
emergency_contact_name      0
emergency_contact_phone     0
dtype: int64

Missing values in Passengers:


passenger_id      0
first_name        0
last_name        10
age               0
gender            0
email             0
phone             0
aadhaar_id        0
date_of_birth     0
dtype: int64

In [7]:
print("Duplicate rows:")

print("Flights:", flights.duplicated().sum())
print("Payments:", payments.duplicated().sum())
print("Bookings:", bookings.duplicated().sum())
print("Passengers:", passengers.duplicated().sum())

Duplicate rows:
Flights: 15
Payments: 0
Bookings: 0
Passengers: 0


## 3. Data Cleaning and Preprocessing

The datasets are cleaned by handling duplicate records, missing values, invalid entries, inconsistent categories, and data-quality issues. Quality flags are retained where appropriate so that problematic records are not silently removed.

In [8]:
flights_clean = flights.drop_duplicates().copy()

print("Original flight records:", len(flights))
print("Flight records after removing duplicates:", len(flights_clean))
print("Duplicates removed:", len(flights) - len(flights_clean))

Original flight records: 1020
Flight records after removing duplicates: 1005
Duplicates removed: 15


In [9]:
flights_clean["airline"] = flights_clean["airline"].fillna("Unknown")
flights_clean["airline"] = flights_clean["airline"].replace("UNKNOWN", "Unknown")

print("Airline values after cleaning:")
print(flights_clean["airline"].value_counts())

Airline values after cleaning:
airline
IndiGo       249
SpiceJet     236
Air India    233
Vistara      218
Unknown       69
Name: count, dtype: int64


In [10]:
flights_clean["departure_time"] = pd.to_datetime(flights_clean["departure_time"])
flights_clean["arrival_time"] = pd.to_datetime(flights_clean["arrival_time"])

calculated_duration = (
    flights_clean["arrival_time"] - flights_clean["departure_time"]
)

flights_clean["overnight"] = (
    flights_clean["arrival_time"].dt.date >
    flights_clean["departure_time"].dt.date
)

flights_clean["time_quality"] = "Valid"

flights_clean.loc[
    calculated_duration < pd.Timedelta(0),
    "time_quality"
] = "Invalid"

print("Invalid flight timings:", (flights_clean["time_quality"] == "Invalid").sum())
print("Overnight flights:", flights_clean["overnight"].sum())

Invalid flight timings: 1
Overnight flights: 122


In [11]:
provided_duration = flights_clean["duration"].apply(
    lambda x: pd.Timedelta(
        hours=x.hour,
        minutes=x.minute,
        seconds=x.second
    )
)

duration_difference = abs(
    provided_duration - calculated_duration
)

flights_clean["duration_mismatch"] = (
    duration_difference > pd.Timedelta(minutes=5)
)

print(
    "Duration mismatches:",
    flights_clean["duration_mismatch"].sum()
)

Duration mismatches: 1


In [12]:
flights_clean["overall_quality"] = "Valid"

flights_clean.loc[
    (flights_clean["time_quality"] == "Invalid") |
    (flights_clean["duration_mismatch"]),
    "overall_quality"
] = "Anomaly"

print("Overall flight quality:")
print(flights_clean["overall_quality"].value_counts())

Overall flight quality:
overall_quality
Valid      1004
Anomaly       1
Name: count, dtype: int64


In [13]:
flights_clean["flight_id_valid"] = flights_clean["flight_id"].astype(str).str.match(
    r"^[A-Z0-9]{2}\d{3}$"
)

print("Valid flight IDs:", flights_clean["flight_id_valid"].sum())
print("Invalid flight IDs:", (~flights_clean["flight_id_valid"]).sum())

Valid flight IDs: 1005
Invalid flight IDs: 0


In [14]:
bookings_clean = bookings.copy()

bookings_clean["status"] = bookings_clean["status"].fillna("Unknown")
bookings_clean["status"] = bookings_clean["status"].replace(
    "INVALID", "Unknown"
)

bookings_clean["status_quality"] = bookings_clean["status"].map({
    "CONFIRMED": "Valid",
    "CANCELLED": "Valid",
    "PENDING": "Valid",
    "Unknown": "Missing/Invalid"
})

print("Booking status:")
print(bookings_clean["status"].value_counts())

print("\nStatus quality:")
print(bookings_clean["status_quality"].value_counts())

Booking status:
status
CONFIRMED    320
CANCELLED    314
PENDING      291
Unknown       75
Name: count, dtype: int64

Status quality:
status_quality
Valid              925
Missing/Invalid     75
Name: count, dtype: int64


In [15]:
passengers_clean = passengers.copy()

passengers_clean["last_name"] = passengers_clean["last_name"].fillna("Unknown")

passengers_clean["last_name_quality"] = "Valid"
passengers_clean.loc[
    passengers_clean["last_name"] == "Unknown",
    "last_name_quality"
] = "Missing"

print("Last name quality:")
print(passengers_clean["last_name_quality"].value_counts())

Last name quality:
last_name_quality
Valid      1029
Missing      10
Name: count, dtype: int64


In [16]:
payments_clean = payments.copy()

payments_clean["amount"] = pd.to_numeric(
    payments_clean["amount"].replace("INVALID", pd.NA),
    errors="coerce"
)

payments_clean["amount_quality"] = payments_clean["amount"].isna().map({
    True: "Missing/Invalid",
    False: "Valid"
})

print("Payment amount quality:")
print(payments_clean["amount_quality"].value_counts())

Payment amount quality:
amount_quality
Valid              922
Missing/Invalid     78
Name: count, dtype: int64


In [17]:
missing_flights = ~bookings_clean["flight_id"].isin(
    flights_clean["flight_id"]
)

missing_passengers = ~bookings_clean["passenger_id"].isin(
    passengers_clean["passenger_id"]
)

missing_bookings = ~payments_clean["booking_id"].isin(
    bookings_clean["booking_id"]
)

print("Bookings with missing Flight IDs:", missing_flights.sum())
print("Bookings with missing Passenger IDs:", missing_passengers.sum())
print("Payments with missing Booking IDs:", missing_bookings.sum())

Bookings with missing Flight IDs: 0
Bookings with missing Passenger IDs: 0
Payments with missing Booking IDs: 0


In [18]:
flights_clean["duplicate_flight_id"] = flights_clean["flight_id"].duplicated(
    keep=False
)

print(
    "Records with duplicate Flight IDs:",
    flights_clean["duplicate_flight_id"].sum()
)

print(
    "Unique duplicated Flight IDs:",
    flights_clean.loc[
        flights_clean["duplicate_flight_id"],
        "flight_id"
    ].nunique()
)

Records with duplicate Flight IDs: 2
Unique duplicated Flight IDs: 1


In [19]:
print("Flight Data Quality Summary")
print("---------------------------")

print("Total flight records:", len(flights_clean))
print("Valid records:", (flights_clean["overall_quality"] == "Valid").sum())
print("Anomalous records:", (flights_clean["overall_quality"] == "Anomaly").sum())
print("Overnight flights:", flights_clean["overnight"].sum())
print("Duplicate Flight ID records:", flights_clean["duplicate_flight_id"].sum())

Flight Data Quality Summary
---------------------------
Total flight records: 1005
Valid records: 1004
Anomalous records: 1
Overnight flights: 122
Duplicate Flight ID records: 2


In [20]:
output_folder = Path("../cleaned_data")
output_folder.mkdir(exist_ok=True)

flights_clean.to_csv(output_folder / "flights_clean.csv", index=False)
bookings_clean.to_csv(output_folder / "bookings_clean.csv", index=False)
passengers_clean.to_csv(output_folder / "passengers_clean.csv", index=False)
payments_clean.to_csv(output_folder / "payments_clean.csv", index=False)

print("Cleaned datasets saved successfully.")

Cleaned datasets saved successfully.


## 4. Database Creation

The cleaned datasets are stored in a SQLite database to provide a structured and queryable storage layer for further analysis. The database contains separate tables for flights, bookings, passengers, and payments.

In [21]:
import sqlite3

database_path = "../airlines.db"

conn = sqlite3.connect(database_path)

flights_clean.to_sql("flights", conn, if_exists="replace", index=False)
bookings_clean.to_sql("bookings", conn, if_exists="replace", index=False)
passengers_clean.to_sql("passengers", conn, if_exists="replace", index=False)
payments_clean.to_sql("payments", conn, if_exists="replace", index=False)

print("SQLite database created successfully.")
print("Tables: flights, bookings, passengers, payments")

SQLite database created successfully.
Tables: flights, bookings, passengers, payments


## 5. Star Schema Design

A star-schema-based analytical model is created from the cleaned database tables. Passenger and flight information is organized as dimension tables, while booking and payment information is organized as fact tables.

The model supports analysis of bookings, flights, passengers, routes, and payments while keeping the analytical structure simple and efficient.

In [22]:
cursor = conn.cursor()

cursor.execute("DROP TABLE IF EXISTS dim_passenger")
cursor.execute("DROP TABLE IF EXISTS dim_flight")
cursor.execute("DROP TABLE IF EXISTS fact_booking")
cursor.execute("DROP TABLE IF EXISTS fact_payment")

cursor.execute("""
CREATE TABLE dim_passenger AS
SELECT passenger_id, age, gender
FROM passengers
""")

cursor.execute("""
CREATE TABLE dim_flight AS
SELECT
    flight_id,
    airline,
    source,
    destination,
    departure_time,
    arrival_time,
    duration,
    overnight,
    time_quality,
    duration_mismatch,
    overall_quality,
    flight_id_valid,
    duplicate_flight_id
FROM flights
""")

cursor.execute("""
CREATE TABLE fact_booking AS
SELECT
    booking_id,
    passenger_id,
    flight_id,
    booking_date,
    status,
    status_quality
FROM bookings
""")

cursor.execute("""
CREATE TABLE fact_payment AS
SELECT
    payment_id,
    booking_id,
    amount,
    payment_method,
    amount_quality
FROM payments
""")

conn.commit()

print("Star schema created successfully.")

Star schema created successfully.


In [23]:
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name",
    conn
)

display(tables)

,name
0,bookings
1,dim_flight
2,dim_passenger
3,fact_booking
4,fact_payment
5,flights
6,passengers
7,payments


## 6. SQL Analysis and Key Performance Indicators

SQL queries are used to calculate important performance indicators and generate analytical datasets. The analysis covers flight performance, booking status, cancellation rate, routes, payment activity, and data quality.

In [24]:
flight_kpi = pd.read_sql_query("""
SELECT
    COUNT(*) AS total_flights,
    SUM(CASE WHEN overall_quality = 'Valid' THEN 1 ELSE 0 END) AS valid_flights,
    SUM(CASE WHEN overall_quality = 'Anomaly' THEN 1 ELSE 0 END) AS anomalous_flights,
    SUM(CASE WHEN overnight = 1 THEN 1 ELSE 0 END) AS overnight_flights
FROM flights
""", conn)

display(flight_kpi)

,total_flights,valid_flights,anomalous_flights,overnight_flights
0,1005,1004,1,122


In [25]:
booking_kpi = pd.read_sql_query("""
SELECT
    COUNT(*) AS total_bookings,
    SUM(CASE WHEN status = 'CONFIRMED' THEN 1 ELSE 0 END) AS confirmed_bookings,
    SUM(CASE WHEN status = 'CANCELLED' THEN 1 ELSE 0 END) AS cancelled_bookings,
    SUM(CASE WHEN status = 'PENDING' THEN 1 ELSE 0 END) AS pending_bookings,
    SUM(CASE WHEN status = 'Unknown' THEN 1 ELSE 0 END) AS unknown_bookings
FROM bookings
""", conn)

display(booking_kpi)

,total_bookings,confirmed_bookings,cancelled_bookings,pending_bookings,unknown_bookings
0,1000,320,314,291,75


In [26]:
cancellation_rate = pd.read_sql_query("""
SELECT
    ROUND(
        100.0 * SUM(CASE WHEN status = 'CANCELLED' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS cancellation_rate
FROM bookings
""", conn)

display(cancellation_rate)

,cancellation_rate
0,31.4


In [27]:
airline_performance = pd.read_sql_query("""
SELECT
    airline,
    COUNT(*) AS total_flights,
    ROUND(AVG(
        (strftime('%s', arrival_time) - strftime('%s', departure_time)) / 60.0
    ), 2) AS average_duration_minutes
FROM flights
WHERE overall_quality = 'Valid'
GROUP BY airline
ORDER BY total_flights DESC
""", conn)

display(airline_performance)

,airline,total_flights,average_duration_minutes
0,IndiGo,249,167.70
1,SpiceJet,235,163.61
2,Air India,233,163.15
3,Vistara,218,162.68
4,Unknown,69,166.07


In [28]:
route_performance = pd.read_sql_query("""
SELECT
    source,
    destination,
    COUNT(*) AS total_flights
FROM flights
WHERE overall_quality = 'Valid'
GROUP BY source, destination
ORDER BY total_flights DESC
LIMIT 10
""", conn)

display(route_performance)

,source,destination,total_flights
0,BOM,CCU,90
1,CCU,DEL,72
2,MAA,BLR,65
3,BLR,BOM,60
4,HYD,MAA,57
5,DEL,HYD,54
6,HYD,DEL,42
7,BOM,DEL,39
8,CCU,BOM,33
9,DEL,BLR,29


In [29]:
booking_status = pd.read_sql_query("""
SELECT
    status,
    COUNT(*) AS total_bookings,
    ROUND(
        100.0 * COUNT(*) / (SELECT COUNT(*) FROM bookings),
        2
    ) AS percentage
FROM bookings
GROUP BY status
ORDER BY total_bookings DESC
""", conn)

display(booking_status)

,status,total_bookings,percentage
0,CONFIRMED,320,32.0
1,CANCELLED,314,31.4
2,PENDING,291,29.1
3,Unknown,75,7.5


In [30]:
booking_trend = pd.read_sql_query("""
SELECT
    DATE(booking_date) AS booking_date,
    COUNT(*) AS total_bookings,
    SUM(CASE WHEN status = 'CONFIRMED' THEN 1 ELSE 0 END) AS confirmed,
    SUM(CASE WHEN status = 'CANCELLED' THEN 1 ELSE 0 END) AS cancelled,
    SUM(CASE WHEN status = 'PENDING' THEN 1 ELSE 0 END) AS pending
FROM bookings
GROUP BY DATE(booking_date)
ORDER BY DATE(booking_date)
""", conn)

display(booking_trend.head(10))
print("Number of booking dates:", len(booking_trend))

,booking_date,total_bookings,confirmed,cancelled,pending
0,2025-04-17,2,0,2,0
1,2025-04-18,3,0,2,1
2,2025-04-19,1,0,0,1
3,2025-04-20,3,2,0,0
4,2025-04-21,5,0,1,3
5,2025-04-22,3,1,0,1
6,2025-04-23,1,0,1,0
7,2025-04-24,7,2,2,3
8,2025-04-25,5,4,1,0
9,2025-04-26,1,1,0,0


Number of booking dates: 342


In [31]:
payment_summary = pd.read_sql_query("""
SELECT
    COUNT(*) AS total_payments,
    SUM(CASE WHEN amount_quality = 'Valid' THEN 1 ELSE 0 END) AS valid_payments,
    SUM(CASE WHEN amount_quality = 'Missing/Invalid' THEN 1 ELSE 0 END) AS invalid_payments,
    ROUND(SUM(amount), 2) AS total_revenue
FROM payments
""", conn)

display(payment_summary)

,total_payments,valid_payments,invalid_payments,total_revenue
0,1000,922,78,7385142.98


In [32]:
payment_statistics = pd.read_sql_query("""
SELECT
    ROUND(AVG(amount), 2) AS average_payment,
    ROUND(MIN(amount), 2) AS minimum_payment,
    ROUND(MAX(amount), 2) AS maximum_payment
FROM payments
WHERE amount_quality = 'Valid'
""", conn)

display(payment_statistics)

,average_payment,minimum_payment,maximum_payment
0,8009.92,1002.59,14992.95


In [33]:
payment_method_analysis = pd.read_sql_query("""
SELECT
    payment_method,
    COUNT(*) AS total_payments,
    SUM(CASE WHEN amount_quality = 'Valid' THEN 1 ELSE 0 END) AS valid_payments,
    ROUND(SUM(amount), 2) AS total_revenue,
    ROUND(AVG(amount), 2) AS average_payment
FROM payments
GROUP BY payment_method
ORDER BY total_revenue DESC
""", conn)

display(payment_method_analysis)

,payment_method,total_payments,valid_payments,total_revenue,average_payment
0,UPI,358,331,2618686.77,7911.44
1,CARD,329,300,2400812.04,8002.71
2,NETBANKING,313,291,2365644.17,8129.36


In [34]:
data_quality = pd.DataFrame({
    "Data Quality Check": [
        "Exact duplicate flight records",
        "Invalid flight timing",
        "Duration mismatch",
        "Duplicate Flight IDs",
        "Missing/Invalid booking status",
        "Missing passenger last name",
        "Missing/Invalid payment amount"
    ],
    "Affected Records": [
        len(flights) - len(flights_clean),
        (flights_clean["time_quality"] == "Invalid").sum(),
        flights_clean["duration_mismatch"].sum(),
        flights_clean["duplicate_flight_id"].sum(),
        (bookings_clean["status_quality"] == "Missing/Invalid").sum(),
        (passengers_clean["last_name_quality"] == "Missing").sum(),
        (payments_clean["amount_quality"] == "Missing/Invalid").sum()
    ]
})

display(data_quality)

,Data Quality Check,Affected Records
0,Exact duplicate flight records,15
1,Invalid flight timing,1
2,Duration mismatch,1
3,Duplicate Flight IDs,2
4,Missing/Invalid booking status,75
5,Missing passenger last name,10
6,Missing/Invalid payment amount,78


In [35]:
booking_trend.to_csv(output_folder / "booking_trend.csv", index=False)
airline_performance.to_csv(output_folder / "airline_performance.csv", index=False)
route_performance.to_csv(output_folder / "route_performance.csv", index=False)
booking_status.to_csv(output_folder / "booking_status.csv", index=False)
data_quality.to_csv(output_folder / "data_quality.csv", index=False)

print("All analytical datasets saved successfully.")

All analytical datasets saved successfully.


## 7. Final Results

The completed pipeline successfully processed the airline datasets and produced cleaned datasets, a SQLite database, a star-schema-based analytical model, and SQL-based analytical outputs.

### Key Results

- 1,005 flight records were retained after removing 15 exact duplicate records.
- 1,000 booking records were processed.
- 1,039 passenger records were processed.
- 1,000 payment records were processed.
- 1,004 flight records were classified as valid and 1 record was flagged as an anomaly.
- 122 flights were identified as overnight flights.
- The booking cancellation rate was 31.4%.
- 922 payment records contained valid payment amounts.
- The total revenue from valid payment amounts was approximately ₹73.85 lakh.
- Analytical datasets were generated for booking trends, airline performance, route performance, booking status, and data quality.

In [36]:
print("FINAL PROJECT VALIDATION")
print("------------------------")

print("Flights:", len(flights_clean))
print("Bookings:", len(bookings_clean))
print("Passengers:", len(passengers_clean))
print("Payments:", len(payments_clean))

print("\nAnalytical datasets:")
print("Booking trend:", len(booking_trend))
print("Airline performance:", len(airline_performance))
print("Top routes:", len(route_performance))
print("Booking status:", len(booking_status))
print("Data quality records:", len(data_quality))

FINAL PROJECT VALIDATION
------------------------
Flights: 1005
Bookings: 1000
Passengers: 1039
Payments: 1000

Analytical datasets:
Booking trend: 342
Airline performance: 5
Top routes: 10
Booking status: 4
Data quality records: 7


In [37]:
conn.close()

print("Database connection closed successfully.")
print("Airlines data engineering pipeline completed successfully.")

Database connection closed successfully.
Airlines data engineering pipeline completed successfully.


## 8. Conclusion

The project demonstrates an end-to-end airline data engineering workflow starting from raw Excel data and progressing through data cleaning, quality validation, database storage, star schema design, SQL analysis, and preparation for business intelligence visualization.

The resulting pipeline provides a reproducible process for transforming the raw airline data into structured and analysis-ready datasets. The outputs can be further used in Power BI to develop an interactive dashboard for monitoring flights, bookings, routes, payments, and data quality.

In [39]:
from pathlib import Path

gitignore_content = """# Raw and sensitive data
data/raw/
cleaned_data/

# Database files
*.db
*.sqlite

# Jupyter temporary files
.ipynb_checkpoints/

# Python cache
__pycache__/
*.pyc

# macOS files
.DS_Store
"""

Path("../.gitignore").write_text(gitignore_content)

print(".gitignore created successfully")

.gitignore created successfully


In [40]:
from pathlib import Path
import shutil

shutil.move("../AIRLINE USE CASE.docx", "../docs/AIRLINE USE CASE.docx")
shutil.move("../Airlines_Data_Dictionary.xlsx", "../docs/Airlines_Data_Dictionary.xlsx")

print("Files moved to docs successfully")

Files moved to docs successfully
